# D3.10 · Hunting in agent telemetry

**Function D — The Agentic SOC → Understand — Correlation, Intel and the Hunt**

Builds on **[D3.9 · Third-party threat intelligence, and the tactics it names](https://spbreed.github.io/cyber-commons/lessons/D3.9.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Hypothesis-first hunting over agent telemetry: stating a falsifiable hypothesis, naming the population before running it, scoring what it caught against what it missed, and deciding to promote, tune or discard.

**Why a security engineer needs it.** Detections encode behaviour somebody already understood. Everything outside them is invisible, and the agent behaviour worth catching is usually behaviour nobody had thought to write a rule for. The discipline is in the scoring: the hypothesis that feels most obviously right is often a description of normal work, and shipping it costs a quarter of everyone's attention.

| | |
|---|---|
| **Day 0 — why** | Everything not covered by a rule is invisible, and the rules were written against behaviour somebody already understood. |
| **Day 1 — how** | State a falsifiable hypothesis, name the population before running it, then promote, tune or discard. |
| **Day 2 — measure** | Precision of the hunt: a hypothesis matching fourteen runs to find two, because twelve are the nightly batch, has bought nothing. |

## 1 · The hook

The alert queue has been quiet for a fortnight and nobody believes it. Hunting is the pass that finds what no rule was written for — and the hypothesis that feels most obviously right is usually a description of the overnight batch doing its job.

> **At CyberTravels.** The corpus is CyberTravels' agent runs — the Workflow Agent and the RAG Advisor, forty runs across a fortnight. The overnight batch that wrecks the working-hours hypothesis is CyberTravels' own nightly reconciliation, which is exactly the kind of legitimate oddity that makes an obvious hunt useless in a real estate.

## 2 · The framework

```
   DETECTION                          HUNTING

   behaviour  ->  rule  ->  alert     hypothesis
   (already understood)                    |
                                           v
                                      run over stored traces
                                           |
                          +----------------+----------------+
                          v                v                v
                      promote            tune            discard
                   (with an FP rate)  (fires on the      (describes
                                       job itself)        nothing)

   the middle branch is where quarters go: a hypothesis that matches
   fourteen runs to find two, because twelve are the nightly batch
```

A detection encodes a behaviour somebody already understood. Hunting goes the
other way: you state a hypothesis about behaviour that *would* be suspicious,
run it over stored traces, and find out whether it happens.

Over agent telemetry the hypothesis is not the endpoint one. It is not "has
something malicious executed" — it is **"has an agent done something its stated
purpose does not explain"**. A tool it never needs. An hour it never runs. A
volume no task requires.

The failure mode is specific and expensive: a hypothesis that describes the job
rather than the anomaly. It fires constantly, everyone stops reading it, and the
quarter is gone.

> **Anchor → D1.0.** Hunting is the only pass that shortens discover for behaviour no rule was written for, and it spends real time to do it. Hence the scoring: a hypothesis matching fourteen runs to find two, because twelve are the nightly batch, has bought nothing.

## 3 · A hypothesis is a sentence with a subject

"A workflow agent called a payments tool outside a booking flow" can be
falsified. "Look for anomalies" cannot — it is a wish.

And it needs a population before it runs. Without one, a hit rate has no
denominator, and five hits is not a result until you know five out of how
many.

## 4 · Three hypotheses, scored

Watch the middle row. It matches fourteen runs to find two, because twelve of the matches are CyberTravels' overnight batch doing exactly its job.

### The skill — [`skills/detection/agent-telemetry-hunt/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-telemetry-hunt/SKILL.md)

```yaml
name: agent-telemetry-hunt
description: >-
  Run a hypothesis-first hunt over agent traces and score what it catches
  against what it misses. Use when looking for agent behaviour no rule was
  written for, when deciding whether a hunt finding should become a detection,
  or when a clean detection surface needs testing from the other direction.
allowed-tools: Read, Grep, Glob
```

# Hunting is the pass that finds what nobody wrote a rule for

A detection encodes a behaviour somebody already understood. A hunt goes the
other way: you state a hypothesis about behaviour that *would* be suspicious,
run it over stored traces, and find out whether it happens.

Over agent telemetry the hypothesis is not the endpoint one. It is not "has
something malicious executed". It is **"has an agent done something its stated
purpose does not explain"** — a tool it never needs, an hour it never runs, a
volume no task requires.

## When to use this

After detection coverage is written, not before. A hunt over a surface with no
rules just re-finds what a rule would have caught cheaply. Run one when the
alert queue has gone quiet and you do not believe it, when an agent's scope
changed, and on a standing cadence for the agents that hold the most authority.

## Step-by-step

**1 — State the hypothesis as a sentence with a subject.** "A workflow agent
called a payments tool outside a booking flow." Not "look for anomalies" —
that is a wish, and it cannot be falsified.

**2 — Name the population before you run it.** Which agents, which window.
Without it, a hit rate has no denominator and you cannot tell a rare event from
a common one you sampled badly.

**3 — Run it and count both sides.** What matched, and what should have matched
and did not. A hunt with only hits is a demo.

**4 — Decide the outcome explicitly: promote, tune, or discard.** A hypothesis
that fires on normal work is not a finding, it is a description of the job.

**5 — Promote to a detection only with a false-positive number attached.**
That is the handover to `detection-rule-synthesis`.

## Example

**Input** — a labelled trace corpus committed in
[`scripts/agent_telemetry_hunt.py`](scripts/agent_telemetry_hunt.py), with
three hypotheses.

**Output** — the opening lines of a real run:

```
corpus: 40 agent runs, 6 of them labelled anomalous

hypothesis                       matched  tp  fp  precision  recall
tool outside declared scope            3   3   0       1.00    0.50
run outside working hours             14   2  12       0.14    0.33
volume above task ceiling              2   2   0       1.00    0.33
```

The middle row is the lesson. Twelve of its fourteen matches are the overnight
batch doing its job, so it is a description of normal work wearing a
hypothesis' clothes. Tune it or discard it — do not ship it.

## Output contract

```json
{
  "corpus": {"runs": 0, "anomalous": 0},
  "hypotheses": [{"name": "str", "matched": 0, "true_positives": 0,
                  "false_positives": 0, "precision": 0.0, "recall": 0.0,
                  "outcome": "promote|tune|discard"}]
}
```

## Common edge cases

- **The hypothesis describes the job.** "Agent called a tool" matches
  everything. If precision is near the base rate, you have described normal.
- **The window excludes the behaviour.** Hunting 24 hours for something that
  happens monthly returns nothing and proves nothing.
- **Labels are the hunt's own output.** Scoring a hunt against findings the
  same hunt produced measures nothing.

## Failure modes

- **Hunting without a denominator.** Five hits is not a result until you know
  five out of how many.
- **Promoting on precision alone.** A rule that fires on one known case and
  nothing else has perfect precision and no value.
- **Never discarding.** A hunt library where nothing is ever retired becomes a
  second alert queue with worse rules.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-telemetry-hunt/scripts/agent_telemetry_hunt.py
SCRIPT = "skills/detection/agent-telemetry-hunt/scripts/agent_telemetry_hunt.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.5 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Three hypotheses scored on precision and recall against a labelled corpus, each ending in promote, tune or discard — and the working-hours hypothesis rejected at precision 0.14.

## Your turn

Write a fourth hypothesis for CyberTravels and score it. If its precision is near the base rate, you have described normal work.

## Where this leaves you

**What you can do now.** You can end the understand interval honestly: the investigator bounded before it starts, an alert carrying the fields agent triage needs, a trace where the first theory was abandoned in the open, scope walked along the delegation graph, coordination that exists only in the population, third-party intel that had to become a rule to count, and a hunt scored on precision.

**What you still cannot do.** You know what happened and you have not stopped it. Every lever you might pull is still chosen in the moment by whoever is awake, so the contain interval is whatever that person's night is like.

**Chapter D4 makes contain a number you set in advance: the remediation policy that decides what may happen without asking, and the three runbook tiers that policy produces.**

---

**Next → [D4.1 · Remediation policy — what may be done without asking](https://spbreed.github.io/cyber-commons/lessons/D4.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D3.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D3.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*